# 01 – Decision Trees & Random Forests

Topics covered:
1. Decision Tree – how it works, visualisation
2. Overfitting & pruning (`max_depth`)
3. Random Forest – ensemble of trees
4. Feature importance

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set_theme(style='whitegrid')
np.random.seed(42)

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target
feature_names = wine.feature_names
class_names   = wine.target_names

print('Shape:', X.shape)
print('Classes:', class_names)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1. Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_tr, y_tr)

print('Train accuracy (no depth limit):', accuracy_score(y_tr, dt.predict(X_tr)))
print('Test  accuracy (no depth limit):', accuracy_score(y_te, dt.predict(X_te)))

In [ ]:
# Visualise the tree (limited depth for readability)
plt.figure(figsize=(18, 6))
plot_tree(dt, max_depth=3, feature_names=feature_names,
          class_names=class_names, filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree (max_depth=3 for display)')
plt.show()

In [ ]:
# Overfitting analysis: vary max_depth
depths      = range(1, 15)
train_accs  = []
test_accs   = []

for d in depths:
    dt_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_d.fit(X_tr, y_tr)
    train_accs.append(accuracy_score(y_tr, dt_d.predict(X_tr)))
    test_accs.append(accuracy_score(y_te, dt_d.predict(X_te)))

plt.figure(figsize=(8, 5))
plt.plot(depths, train_accs, label='Train', marker='o')
plt.plot(depths, test_accs,  label='Test',  marker='s')
plt.xlabel('max_depth'); plt.ylabel('Accuracy')
plt.title('Overfitting: Train vs Test Accuracy')
plt.legend()
plt.show()

## 2. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

y_pred_rf = rf.predict(X_te)
print('Random Forest Test Accuracy:', accuracy_score(y_te, y_pred_rf))
print(classification_report(y_te, y_pred_rf, target_names=class_names))

In [ ]:
# Cross-validation
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f'CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
# Feature importance
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Random Forest – Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 3. Key Takeaways

| Concept | Notes |
|---------|-------|
| Decision Tree | Interpretable, but prone to overfitting |
| `max_depth` pruning | Controls model complexity |
| Random Forest | Bagging of trees → lower variance, better generalisation |
| Feature importance | Gini / impurity-based; can be misleading for correlated features |

**Next:** `02_SVM.ipynb`